# MethodThinker Colab Training Notebook

> **云端GPU训练模板** - 专为Google Colab T4 GPU优化

**使用说明**:
1. 依次运行所有Cell
2. 确保GPU已启用（运行时 → 更改运行时类型 → T4 GPU）
3. 训练完成后自动保存到Google Drive

---

## 1. GPU检查与设置

验证GPU状态，确保T4 GPU已启用。

In [ ]:
# GPU状态检查
import torch

print("="*50)
print("GPU状态检查")
print("="*50)

if not torch.cuda.is_available():
    print("⚠️ GPU未启用！")
    print("请执行以下步骤：")
    print("  1. 点击菜单 '运行时' → '更改运行时类型'")
    print("  2. 选择 'T4 GPU'")
    print("  3. 点击 '保存' 重新连接")
else:
    print(f"✅ GPU已启用")
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"支持BF16: {torch.cuda.is_bf16_supported()}")

print("="*50)

# 如果GPU未启用，停止后续执行
assert torch.cuda.is_available(), "请启用GPU后重新运行此笔记本"

In [ ]:
# 查看详细GPU信息
!nvidia-smi

## 2. 环境设置

克隆项目并安装依赖。

In [ ]:
# 克隆项目
# 请替换为您的实际仓库地址
REPO_URL = "https://github.com/your-repo/method-thinker.git"

!git clone {REPO_URL}
%cd method-thinker

print("✅ 项目已克隆")

In [ ]:
# 设置HuggingFace镜像（国内加速）
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print("✅ HuggingFace镜像已设置")

# 安装核心依赖
!pip install -q torch>=2.1.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.40.0 accelerate>=0.25.0 peft>=0.7.0 trl>=0.7.0 datasets>=2.14.0
!pip install -q pyyaml pydantic numpy scipy sympy pandas tqdm rich python-dotenv

print("✅ 依赖已安装")

In [ ]:
# 验证环境
import torch
import transformers
import accelerate
import peft
import trl

print("="*50)
print("环境验证报告")
print("="*50)
print(f"PyTorch版本: {torch.__version__}")
print(f"Transformers版本: {transformers.__version__}")
print(f"Accelerate版本: {accelerate.__version__}")
print(f"PEFT版本: {peft.__version__}")
print(f"TRL版本: {trl.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"支持BF16: {torch.cuda.is_bf16_supported()}")
print("="*50)
print("✅ 环境验证完成")

## 3. Google Drive挂载

挂载Drive用于保存训练模型。

In [ ]:
# 挂载Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# 创建模型保存目录
SAVE_DIR = '/content/drive/MyDrive/method-thinker-models'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"✅ Drive已挂载，模型将保存到: {SAVE_DIR}")

## 4. 模型加载

加载基座模型和分词器。

In [ ]:
# 加载基座模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"

print(f"加载模型: {MODEL_NAME}")
print("这可能需要几分钟...")

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

# 设置pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("✅ 模型加载完成")

# 显示模型信息
print(f"\n模型参数量: {model.num_parameters() / 1e9:.2f}B")
print(f"模型设备: {model.device}")

## 5. LoRA配置

应用LoRA节省显存，实现高效微调。

In [ ]:
# 应用LoRA配置
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                              # LoRA秩
    lora_alpha=32,                     # LoRA alpha
    lora_dropout=0.05,                 # Dropout率
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)

# 显示可训练参数
model.print_trainable_parameters()

print("✅ LoRA配置已应用")

## 6. 数据加载

加载训练数据和知识库。

In [ ]:
# 加载知识库
import yaml

KB_PATH = "data/methodology_kb/v0/math_methods.yaml"

if os.path.exists(KB_PATH):
    with open(KB_PATH, 'r', encoding='utf-8') as f:
        kb_data = yaml.safe_load(f)
    print(f"✅ 知识库已加载: {len(kb_data.get('methods', []))} 个方法")
else:
    print(f"⚠️ 知识库文件不存在: {KB_PATH}")
    print("使用默认知识库")
    kb_data = {'methods': []}

# 检查训练数据
TRAIN_PATH = "data/train_data/train.json"

if os.path.exists(TRAIN_PATH):
    print(f"✅ 训练数据存在: {TRAIN_PATH}")
else:
    print(f"⚠️ 训练数据不存在: {TRAIN_PATH}")
    print("将使用示例数据进行演示")

In [ ]:
# 创建示例训练数据（如果实际数据不存在）
import json

if not os.path.exists(TRAIN_PATH):
    os.makedirs("data/train_data", exist_ok=True)
    
    # 创建示例数据
    sample_data = [
        {
            "problem_id": "sample_001",
            "problem": "求方程 x^2 - 5x + 6 = 0 的解",
            "problem_type": "二次方程",
            "difficulty": 1,
            "candidate_methods": [
                {"method_name": "因式分解法", "applicability_score": 0.9},
                {"method_name": "公式法", "applicability_score": 0.8}
            ],
            "selected_method": "因式分解法",
            "selection_reasoning": "方程可因式分解为(x-2)(x-3)=0",
            "solution_steps": [
                "将方程分解: x^2 - 5x + 6 = (x-2)(x-3) = 0",
                "得 x-2=0 或 x-3=0",
                "解得 x=2 或 x=3"
            ],
            "reflection": "使用因式分解法简洁高效，答案正确。"
        },
        {
            "problem_id": "sample_002",
            "problem": "计算 lim(x→0) sin(x)/x",
            "problem_type": "极限计算",
            "difficulty": 2,
            "candidate_methods": [
                {"method_name": "洛必达法则", "applicability_score": 0.85},
                {"method_name": "泰勒展开", "applicability_score": 0.9}
            ],
            "selected_method": "泰勒展开",
            "selection_reasoning": "sin(x)在x=0处可用泰勒展开",
            "solution_steps": [
                "sin(x) ≈ x - x^3/6 + ...",
                "sin(x)/x ≈ 1 - x^2/6 + ...",
                "当x→0时，极限为1"
            ],
            "reflection": "泰勒展开直观展示了极限的本质。"
        }
    ]
    
    # 保存示例数据
    with open(TRAIN_PATH, 'w', encoding='utf-8') as f:
        json.dump(sample_data, f, ensure_ascii=False, indent=2)
    
    # 创建验证数据
    with open("data/train_data/val.json", 'w', encoding='utf-8') as f:
        json.dump(sample_data[:1], f, ensure_ascii=False, indent=2)
    
    print(f"✅ 示例数据已创建: {len(sample_data)} 条")
else:
    # 加载实际数据
    with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
        sample_data = json.load(f)
    print(f"✅ 加载训练数据: {len(sample_data)} 条")

## 7. 开始训练

启动方法论注入训练。

In [ ]:
# 配置训练参数
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="outputs/checkpoints_colab",
    num_train_epochs=3,
    per_device_train_batch_size=4,     # T4显存限制
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,     # 有效batch_size=32
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=50,
    save_steps=200,
    save_total_limit=3,
    bf16=True,                          # T4支持BF16
    gradient_checkpointing=True,        # 节省显存
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=42,
    report_to="none",
)

print("✅ 训练参数已配置")
print(f"\n训练配置:")
print(f"  - 训练轮数: {training_args.num_train_epochs}")
print(f"  - 批大小: {training_args.per_device_train_batch_size}")
print(f"  - 有效批大小: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  - 学习率: {training_args.learning_rate}")
print(f"  - BF16训练: {training_args.bf16}")

In [ ]:
# 准备训练数据集
from datasets import Dataset

def format_training_sample(sample):
    """格式化训练样本"""
    # 构建输入
    candidates_str = "\n".join([
        f"{i+1}. {m['method_name']}（评分：{m['applicability_score']}）"
        for i, m in enumerate(sample['candidate_methods'])
    ])
    
    steps_str = "\n".join(sample['solution_steps'])
    
    input_text = f"""【问题】
{sample['problem']}

【题型】
{sample['problem_type']}

【候选方法】
{candidates_str}

请分析各方法并选择最合适的方法解答问题。
"""
    
    output_text = f"""【方法选择】
选中方法：{sample['selected_method']}

【选择理由】
{sample['selection_reasoning']}

【解答过程】
{steps_str}

【反思与验证】
{sample['reflection']}
"""
    
    return {
        'text': input_text + output_text,
        'input': input_text,
        'output': output_text
    }

# 格式化数据
formatted_data = [format_training_sample(s) for s in sample_data]
train_dataset = Dataset.from_list(formatted_data)

print(f"✅ 数据集已准备: {len(train_dataset)} 条")
print(f"\n示例数据（第一条）:")
print(train_dataset[0]['text'][:500] + "...")

In [ ]:
# 开始训练
from trl import SFTTrainer
import time

print("="*50)
print("开始方法论注入训练")
print("="*50)

# 创建训练器
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    max_seq_length=2048,               # T4显存限制
    packing=False,
)

# 开始训练
start_time = time.time()
train_result = trainer.train()
elapsed_time = time.time() - start_time

print("\n" + "="*50)
print("训练完成")
print("="*50)
print(f"训练耗时: {elapsed_time:.1f} 秒")
print(f"最终损失: {train_result.training_loss:.4f}")
print(f"训练步数: {train_result.global_step}")

## 8. 保存模型

保存训练后的模型到Google Drive。

In [ ]:
# 保存模型
import shutil
from datetime import datetime

# 本地保存
trainer.save_model("outputs/checkpoints_colab/final")
tokenizer.save_pretrained("outputs/checkpoints_colab/final")

print("✅ 模型已保存到本地")

# 复制到Google Drive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_path = f"{SAVE_DIR}/model_{timestamp}"

shutil.copytree("outputs/checkpoints_colab/final", drive_path)

print(f"✅ 模型已保存到Drive: {drive_path}")

# 保存训练报告
report = {
    'timestamp': datetime.now().isoformat(),
    'training_time': elapsed_time,
    'final_loss': train_result.training_loss,
    'global_step': train_result.global_step,
    'config': {
        'model': MODEL_NAME,
        'epochs': training_args.num_train_epochs,
        'batch_size': training_args.per_device_train_batch_size,
        'learning_rate': training_args.learning_rate,
        'lora_r': 16
    }
}

with open(f"{drive_path}/training_report.json", 'w') as f:
    json.dump(report, f, indent=2)

print("✅ 训练报告已保存")

## 9. 模型评估

快速验证训练效果。

In [ ]:
# 测试模型生成
test_problem = "求方程 x^2 - 4x + 3 = 0 的解"

input_text = f"""【问题】
{test_problem}

【题型】
二次方程

【候选方法】
1. 因式分解法（评分：0.9）
2. 公式法（评分：0.8）

请分析各方法并选择最合适的方法解答问题。
"""

# 编码输入
inputs = tokenizer(input_text, return_tensors="pt", max_length=2048, truncation=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# 生成
print("生成解答...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# 解码
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 提取生成部分
if input_text in generated_text:
    generated_text = generated_text[len(input_text):]

print("\n" + "="*50)
print("模型生成结果")
print("="*50)
print(generated_text)

In [ ]:
# 快速评估
from src.training.trainer import MethodThinkerTrainer, TrainingConfig

try:
    # 加载评估脚本
    eval_config = TrainingConfig(
        base_model="outputs/checkpoints_colab/final",
        max_length=2048
    )
    
    eval_trainer = MethodThinkerTrainer(eval_config)
    eval_trainer.load_checkpoint("outputs/checkpoints_colab/final")
    
    # 简单评估
    test_samples = [
        {'problem': 'x^2 - 4x + 3 = 0', 'expected': 'x=1 或 x=3'},
        {'problem': 'x^2 - 5x + 6 = 0', 'expected': 'x=2 或 x=3'}
    ]
    
    print("\n进行快速评估...")
    for sample in test_samples:
        result = eval_trainer._generate_solution(sample['problem'], sample)
        is_correct = eval_trainer._verify_answer(result, sample['expected'])
        print(f"问题: {sample['problem']}")
        print(f"正确: {is_correct}")
        print(f"生成: {result[:200]}...\n")
    
except Exception as e:
    print(f"⚠️ 评估跳过: {e}")
    print("模型已保存，可在本地环境进行完整评估")

## 10. 上传到HuggingFace Hub（可选）

将模型上传到HuggingFace Hub共享。

In [ ]:
# 上传到HuggingFace Hub（需要token）
# 取消注释以下代码并填入您的HF token

# from huggingface_hub import HfApi, login

# # 登录
# HF_TOKEN = "your-huggingface-token"  # 替换为您的token
# login(token=HF_TOKEN)

# # 上传
# api = HfApi()
# REPO_NAME = "your-username/method-thinker-v1"  # 替换为您的repo名

# api.create_repo(repo_id=REPO_NAME, repo_type="model", exist_ok=True)
# api.upload_folder(
#     folder_path="outputs/checkpoints_colab/final",
#     repo_id=REPO_NAME,
#     repo_type="model"
# )

# print(f"✅ 模型已上传: https://huggingface.co/{REPO_NAME}")

print("ℹ️ 上传功能已注释，如需上传请取消注释并填入token")

---

## 总结

训练完成！模型已保存到以下位置：

1. **本地**: `outputs/checkpoints_colab/final`
2. **Google Drive**: `{SAVE_DIR}/model_{timestamp}`

### 下一步

- 在本地环境下载模型进行完整评估
- 上传到HuggingFace Hub共享
- 使用 `scripts/train_sft.py --mode full` 进行完整训练流程

### 相关文档

- [云端训练指南](docs/cloud_training_guide.md)
- [用户指南](docs/user_guide.md)
- [API参考](docs/api_reference.md)

In [ ]:
# 显示最终信息
print("="*50)
print("训练总结")
print("="*50)
print(f"基座模型: {MODEL_NAME}")
print(f"训练方法: LoRA (r=16)")
print(f"训练轮数: {training_args.num_train_epochs}")
print(f"训练耗时: {elapsed_time:.1f} 秒")
print(f"最终损失: {train_result.training_loss:.4f}")
print(f"\n模型保存位置:")
print(f"  本地: outputs/checkpoints_colab/final")
print(f"  Drive: {drive_path}")
print("="*50)
print("\n✅ 所有步骤完成！")